# import 구문은 제일 위에 모아두는 거 기억하시죠?

# API KEY 설정

# 검색어 설정

# url 및 헤더 설정

# api 요청 및 결과값 json 반환
# [힌트] json <-> 파이썬 dictionary 변환
#           import json
#           json.loads(json)

# DB 연결 객체 생성

# SQL 수행을 위한 커서 생성

# API를 통해 받아온 정보를 가지고 INSERT 수행

# 데이터베이스 트랜잭션 처리 (commit -> 반영)

# 커서 및 연결 객체 종료 (반납)

In [5]:
!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 2/2 [dotenv]



In [2]:
import urllib.parse
import urllib.request
import json
import mysql.connector
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
client_id = os.getenv('naver_client_id')
client_secret = os.getenv('naver_secret')

In [5]:
searchText = urllib.parse.quote('라이트 노벨')

In [6]:
url = "https://openapi.naver.com/v1/search/book.json?query="+searchText+"&display=1"

request = urllib.request.Request(url)
request.add_header('X-Naver-Client-Id', client_id)
request.add_header('X-Naver-Client-Secret', client_secret)

In [7]:
response = urllib.request.urlopen(request)
response_body = response.read()
response_body = json.loads(response_body)
response_body

{'lastBuildDate': 'Tue, 30 Jun 2026 15:09:41 +0900',
 'total': 5674,
 'start': 1,
 'display': 1,
 'items': [{'title': '전생했더니 슬라임이었던 건에 대하여 23(완결)(한정판) (S Novel+, 완결 /초판 한정 : 책갈피, 양면커버, PET책갈피 12종, 아크릴 스탠드, 아크릴참)',
   'link': 'https://search.shopping.naver.com/book/catalog/60386025583',
   'image': 'https://shopping-phinf.pstatic.net/main_6038602/60386025583.20260612071218.jpg',
   'author': '후세',
   'discount': '29700',
   'publisher': '소미미디어',
   'pubdate': '20260611',
   'isbn': '9791138444811',
   'description': '[줄거리]\n리무루의 귀환으로 갑자기 의욕이 넘치는 템페스트 진영.\n\n그 와중에 리무루는 베루자도에게 충격적인 사실을 듣게 된다.\n그것은 전투의 근본을 흔들 수 있는 정보였다.\n\n한편, 이바라제도 불안한 움직임을 보이며,\n상황은 점점 더 혼란스러워지고 있다.\n\n그야말로 전력전.\n모두가 아슬아슬한 전투를 강요받는 가운데,\n모든 것은 단 하나의 슬라임에게 맡겨졌다.\n\n대인기 전생 판타지, 드디어 본편 완결!'}]}

In [15]:
total_count = response_body['total']
start_num = 1
loop_count = total_count // 100 + 1
book_list = []

In [16]:
for i in range(loop_count):
    url = "https://openapi.naver.com/v1/search/book.json?query="+searchText+"&display=100&start="+str(start_num)

    request = urllib.request.Request(url)
    request.add_header('X-Naver-Client-Id', client_id)
    request.add_header('X-Naver-Client-Secret', client_secret)
    
    response = urllib.request.urlopen(request)
    response_body = response.read()
    response_body = json.loads(response_body)
    
    book_list += response_body['items']
    start_num += 100
    
    if start_num > 1000:
        break
    
book_list

[{'title': '전생했더니 슬라임이었던 건에 대하여 23(완결)(한정판) (S Novel+, 완결 /초판 한정 : 책갈피, 양면커버, PET책갈피 12종, 아크릴 스탠드, 아크릴참)',
  'link': 'https://search.shopping.naver.com/book/catalog/60386025583',
  'image': 'https://shopping-phinf.pstatic.net/main_6038602/60386025583.20260612071218.jpg',
  'author': '후세',
  'discount': '29700',
  'publisher': '소미미디어',
  'pubdate': '20260611',
  'isbn': '9791138444811',
  'description': '[줄거리]\n리무루의 귀환으로 갑자기 의욕이 넘치는 템페스트 진영.\n\n그 와중에 리무루는 베루자도에게 충격적인 사실을 듣게 된다.\n그것은 전투의 근본을 흔들 수 있는 정보였다.\n\n한편, 이바라제도 불안한 움직임을 보이며,\n상황은 점점 더 혼란스러워지고 있다.\n\n그야말로 전력전.\n모두가 아슬아슬한 전투를 강요받는 가운데,\n모든 것은 단 하나의 슬라임에게 맡겨졌다.\n\n대인기 전생 판타지, 드디어 본편 완결!'},
 {'title': '전생했더니 슬라임이었던 건에 대하여 23(완결) (S Novel+, 완결 / 초판 한정 책갈피, 양면커버)',
  'link': 'https://search.shopping.naver.com/book/catalog/60390110626',
  'image': 'https://shopping-phinf.pstatic.net/main_6039011/60390110626.20260612071210.jpg',
  'author': '후세',
  'discount': '12600',
  'publisher': '소미미디어',
  'pubdate': '20260611',
  'isbn': '9791138444

In [19]:
with mysql.connector.connect( 
    host = 'localhost',
    user = 'root',
    password = '2401',
    database = 'bookdb'
) as connection:
    with connection.cursor() as cursor:
        sql = "insert into naver_book" \
            "(book_title, book_image, author, publisher, isbn, book_description, pub_date)" \
            "values (%s, %s, %s, %s, %s, %s, %s)"
        
        from datetime import datetime

        for book_info in book_list:
            pub_str = book_info.get('pubdate', '')
            
            if pub_str:
                pub_date = datetime.strptime(pub_str, '%Y%m%d').date()
            else:
                pub_date = None
                
            values = (
                book_info['title'],
                book_info['image'],
                book_info['author'],
                book_info['publisher'],
                book_info['isbn'],
                book_info['description'],
                pub_date
            )
            
            cursor.execute(sql, values)
    connection.commit()